# 可视化决策边界

本笔记本演示如何可视化不同分类器的决策边界。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_circles, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

## 1. 生成模拟数据集

In [ ]:
# 生成线性可分数据
X_linear, y_linear = make_classification(
    n_samples=200, n_features=2, n_redundant=0,
    n_informative=2, random_state=42, n_clusters_per_class=1
)

# 生成圆形数据
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=42)

# 生成月亮形数据
X_moons, y_moons = make_moons(n_samples=200, noise=0.1, random_state=42)

datasets = [
    (X_linear, y_linear, '线性可分数据'),
    (X_circles, y_circles, '环形数据'),
    (X_moons, y_moons, '月亮形数据')
]

print(f"线性可分数据形状: {X_linear.shape}")
print(f"环形数据形状: {X_circles.shape}")
print(f"月亮形数据形状: {X_moons.shape}")

## 2. 定义决策边界可视化函数

In [ ]:
def plot_decision_boundary(model, X, y, ax, title):
    """绘制决策边界"""
    # 创建网格
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100)
    )
    
    # 预测网格点
    mesh_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(mesh_points)
    Z = Z.reshape(xx.shape)
    
    # 绘制决策边界
    ax.contourf(xx, yy, Z, alpha=0.3, levels=50)
    ax.contour(xx, yy, Z, colors='black', alpha=0.5, linewidths=1)
    
    # 绘制数据点
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='black')
    ax.set_title(title)
    ax.set_xlabel('特征 1')
    ax.set_ylabel('特征 2')

## 3. 在线性可分数据上可视化

In [ ]:
# 初始化模型
models = [
    ('逻辑回归', LogisticRegression(random_state=42)),
    ('线性SVM', SVC(kernel='linear', random_state=42)),
    ('RBF SVM', SVC(kernel='rbf', random_state=42)),
    ('K近邻 (k=3)', KNeighborsClassifier(n_neighbors=3)),
    ('决策树', DecisionTreeClassifier(random_state=42)),
    ('随机森林', RandomForestClassifier(random_state=42))
]

# 训练并可视化
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, (name, model) in enumerate(models):
    model.fit(X_linear, y_linear)
    plot_decision_boundary(model, X_linear, y_linear, axes[idx], name)

plt.suptitle('线性可分数据 - 不同分类器决策边界', fontsize=16)
plt.tight_layout()
plt.show()

## 4. 在环形数据上可视化

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, (name, model) in enumerate(models):
    model.fit(X_circles, y_circles)
    plot_decision_boundary(model, X_circles, y_circles, axes[idx], name)

plt.suptitle('环形数据 - 不同分类器决策边界', fontsize=16)
plt.tight_layout()
plt.show()

## 5. 在月亮形数据上可视化

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, (name, model) in enumerate(models):
    model.fit(X_moons, y_moons)
    plot_decision_boundary(model, X_moons, y_moons, axes[idx], name)

plt.suptitle('月亮形数据 - 不同分类器决策边界', fontsize=16)
plt.tight_layout()
plt.show()

## 6. SVM 不同核函数对比

In [ ]:
# 不同核函数的 SVM
kernels = [
    ('线性核', 'linear'),
    ('多项式核 (degree=3)', 'poly'),
    ('RBF 核', 'rbf'),
    ('Sigmoid 核', 'sigmoid')
]

for X, y, title in datasets:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for idx, (name, kernel) in enumerate(kernels):
        svm = SVC(kernel=kernel, random_state=42, max_iter=1000)
        try:
            svm.fit(X, y)
            plot_decision_boundary(svm, X, y, axes[idx], name)
        except Exception as e:
            axes[idx].text(0.5, 0.5, f'训练失败\n{str(e)}',
                          ha='center', va='center', transform=axes[idx].transAxes)
            axes[idx].set_title(name)
    
    plt.suptitle(f'{title} - SVM 不同核函数对比', fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. KNN 不同 k 值对比

In [ ]:
# 不同 k 值的 KNN
k_values = [1, 3, 5, 15]

for X, y, title in datasets:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for idx, k in enumerate(k_values):
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X, y)
        plot_decision_boundary(knn, X, y, axes[idx], f'KNN (k={k})')
    
    plt.suptitle(f'{title} - KNN 不同 k 值对比', fontsize=14)
    plt.tight_layout()
    plt.show()